# 01 — Build Master Manifest

**Prerequisites:** notebook 00 must have run successfully.

**Outputs:**
- `data/manifests/master_manifest.csv`  
- `data/processed/notebook_01_report.md`

**Columns filled here:** all auto-fill columns (has_*, tp53_status, tcab1_rna_*, telomerase_*,  
atrx/daxx/smarcal1/top3a_status, include_* flags).  
Columns that remain `unknown` until later phases: tcab1_genomic_status, sv_architecture_17p13_1, alt_status.

In [ ]:
import sys, random, time
from pathlib import Path

random.seed(42)
NOTEBOOK_NAME = '01_build_manifest'
RUN_TIMESTAMP = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ — tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

from src.utils import sha256_file, log_versions
import pandas as pd

vers = log_versions()
print(f'Run: {RUN_TIMESTAMP}')

## Load DepMap data

In [2]:
from pathlib import Path

ROOT = Path.cwd().resolve().parent  # adjust if needed
DEPMAP_DIR = ROOT / "data" / "raw" / "depmap"

In [3]:
from src.manifest_builder import (
    load_depmap_model,
    load_expression_matrix,
    load_damaging_matrix,
    load_cn_gene_matrix,
    load_cn_segments,
    load_fusions,
)

print('Loading DepMap data...')
model_df     = load_depmap_model()
expr_mat     = load_expression_matrix()
damaging_mat = load_damaging_matrix()
cn_gene_mat  = load_cn_gene_matrix()
cn_segs      = load_cn_segments()
fusions      = load_fusions()

os_models = model_df[model_df['OncotreePrimaryDisease'].str.contains('Osteosarcoma', na=False)]
print(f'DepMap osteosarcoma models: {len(os_models)}')
print(f'Expression matrix shape: {expr_mat.shape}')
print(f'Damaging matrix shape: {damaging_mat.shape}')

# Record input checksums
input_checksums = {}
for fname in [
    'Model.csv',
    'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv',
    'OmicsSomaticMutationsMatrixDamaging.csv',
    'OmicsCNGeneWGS.csv',
]:
    p = ROOT / 'data' / 'raw' / 'depmap' / fname
    if p.exists():
        input_checksums[fname] = sha256_file(p)
print('Input checksums recorded.')

Loading DepMap data...
DepMap osteosarcoma models: 35
Expression matrix shape: (1719, 19215)
Damaging matrix shape: (1968, 19578)
Input checksums recorded.


## Build DepMap cell-line manifest rows

In [4]:
from src.manifest_builder import build_depmap_manifest

depmap_manifest = build_depmap_manifest(
    model_df=model_df,
    expr_matrix=expr_mat,
    damaging_matrix=damaging_mat,
    cn_gene_matrix=cn_gene_mat,
    fusions_df=fusions,
    cn_segments_df=cn_segs,
)

print(f'DepMap manifest rows: {len(depmap_manifest)}')
print(f'Columns: {list(depmap_manifest.columns)}')
depmap_manifest[['sample_id','cohort','tp53_status','tcab1_rna_quantile_within_cohort',
                 'tcab1_functional_status','telomerase_status','alt_status']].head(10)

DepMap manifest rows: 35
Columns: ['sample_id', 'cohort', 'sample_type', 'has_rna', 'has_wes', 'has_wgs', 'has_sc', 'has_clinical', 'tp53_status', 'tp53_evidence', 'tcab1_rna_quantile', 'tcab1_rna_value', 'tcab1_rna_quantile_within_cohort', 'tcab1_genomic_status', 'tcab1_genomic_evidence', 'tcab1_functional_status', 'tcab1_functional_evidence', 'tcab1_overall_status', 'tcab1_confidence', 'sv_architecture_17p13_1', 'sv_architecture_evidence', 'alt_status', 'alt_source', 'source_confidence', 'alt_assay_evidence', 'telomerase_status', 'tert_expression', 'telomerase_evidence', 'atrx_status', 'daxx_status', 'smarcal1_status', 'top3a_status', 'purity_estimate', 'purity_quartile', 'include_aim1_primary_strict', 'include_aim1_primary_pragmatic', 'include_aim1_primary_broad', 'include_aim2_strict', 'include_aim2_pragmatic', 'include_aim2_broad', 'include_tp53_sensitivity', 'include_telomerase_negative_secondary', 'notes']


,sample_id,cohort,tp53_status,tcab1_rna_quantile_within_cohort,tcab1_functional_status,telomerase_status,alt_status
0,ACH-000082,DepMap_26Q1,not_altered,1.0,functional_intact_supported,indeterminate,alt_unknown
1,ACH-000359,DepMap_26Q1,not_altered,0.606061,functional_intact_supported,positive,alt_unknown
2,ACH-000364,DepMap_26Q1,not_altered,0.575758,functional_intact_supported,indeterminate,alt_unknown
3,ACH-000410,DepMap_26Q1,not_altered,0.666667,functional_intact_supported,indeterminate,alt_unknown
4,ACH-000613,DepMap_26Q1,altered,0.878788,functional_intact_supported,positive,alt_unknown
5,ACH-000748,DepMap_26Q1,not_altered,0.515152,functional_intact_supported,positive,alt_unknown
6,ACH-001001,DepMap_26Q1,altered,0.848485,functional_intact_supported,positive,alt_unknown
7,ACH-001094,DepMap_26Q1,unknown,unknown,unknown,unknown,alt_unknown
8,ACH-001275,DepMap_26Q1,not_altered,0.636364,functional_intact_supported,positive,alt_unknown
9,ACH-001526,DepMap_26Q1,not_altered,0.69697,functional_intact_supported,indeterminate,alt_unknown


## Load literature cohorts

In [5]:
from src.manifest_builder import (
    load_de_nonneville,
    load_lawlor_pdx_labels,
    load_taglialatela,
    load_chen_2014_pcgp,
    load_chen_2014_tp53_svs,
)

de_non    = load_de_nonneville()
lawlor    = load_lawlor_pdx_labels()
tagliala  = load_taglialatela()
pcgp      = load_chen_2014_pcgp()                 # one row per PCGP patient (Chen 2014)
chen_s4   = load_chen_2014_tp53_svs()             # raw Table S4b TP53 SV breakpoints (GRCh37)

print(f'de_Nonneville rows: {len(de_non)}')
print(f'Lawlor PDX rows:    {len(lawlor)}')
print(f'Taglialatela rows:  {len(tagliala)}')
print(f'PCGP_2014 rows:     {len(pcgp)} (Chen 2014, GRCh37 — classified in-frame; no liftover)')

de_Nonneville rows: 22
Lawlor PDX rows:    17
Taglialatela rows:  7
PCGP_2014 rows:     34 (Chen 2014, GRCh37 — classified in-frame; no liftover)


/Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
/Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/.venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(m

## Merge all cohorts

In [6]:
from src.manifest_builder import merge_literature_rows

manifest = merge_literature_rows(
    depmap_manifest=depmap_manifest,
    de_nonneville_df=de_non,
    lawlor_pdx_df=lawlor,
    taglialatela_df=tagliala,
    pcgp_chen_df=pcgp,
)

print(f'Combined manifest rows: {len(manifest)}')
print(manifest['cohort'].value_counts().to_string())

Combined manifest rows: 115
cohort
DepMap_26Q1           35
PCGP_2014             34
de_Nonneville_2022    22
Lawlor_2026_GCCRI     17
Taglialatela_2026      7


## Pre-fill SV architecture from Lawlor literature calls

In [ ]:
from src.sv_annotator      import prefill_sv_architecture_from_literature
from src.manifest_builder    import prefill_pcgp_sv_architecture, refresh_tcab1_status_from_architecture

# Lawlor PDX architecture pre-fills (Lawlor 2026, Figure 2)
manifest = prefill_sv_architecture_from_literature(manifest)
lawlor_prefilled = manifest[manifest['sv_architecture_evidence'].str.contains('Lawlor', na=False)]
print(f'Pre-filled SV architecture for {len(lawlor_prefilled)} Lawlor PDX samples:')
print(lawlor_prefilled[['sample_id','sv_architecture_17p13_1','sv_architecture_evidence']].to_string())

# Chen 2014 PCGP architecture pre-fills (in-frame GRCh37 classification of Table S4b)
manifest = prefill_pcgp_sv_architecture(manifest, chen_s4)
pcgp_prefilled = manifest[manifest['sv_architecture_evidence'].str.contains('Chen 2014', na=False)]
print(f'\nPre-filled SV architecture for {len(pcgp_prefilled)} PCGP samples (Chen 2014 S4):')
print(pcgp_prefilled[['sample_id','sv_architecture_17p13_1']].head(15).to_string())
print(f'\nPCGP architecture call distribution:')
print(manifest.loc[manifest['cohort']=='PCGP_2014','sv_architecture_17p13_1'].value_counts().to_string())

# Propagate architecture calls → tcab1_genomic_status + tcab1_overall_status.
# Without this step, literature samples with SV architecture (PCGP 'Neither',
# 'TP53-only', 'Both-*'; Lawlor 'Both-*') retain tcab1_genomic_status='unknown'
# and Pass A's intact / disrupted counters see them as missing.
manifest = refresh_tcab1_status_from_architecture(manifest)
print(f'\nPost-refresh tcab1_genomic_status by cohort:')
print(pd.crosstab(manifest['cohort'], manifest['tcab1_genomic_status']).to_string())
print(f'\nPost-refresh tcab1_overall_status by cohort:')
print(pd.crosstab(manifest['cohort'], manifest['tcab1_overall_status']).to_string())

Pre-filled SV architecture for 4 Lawlor PDX samples:
   sample_id sv_architecture_17p13_1                                        sv_architecture_evidence
58      OS33    Both-independent-SVs  Pre-filled from tcab1_pdx_sv_calls.csv (Lawlor 2026, Figure 2)
59      OS37    Both-independent-SVs  Pre-filled from tcab1_pdx_sv_calls.csv (Lawlor 2026, Figure 2)
60     OS39R          Both-single-SV  Pre-filled from tcab1_pdx_sv_calls.csv (Lawlor 2026, Figure 2)
71  0560-000          Both-single-SV  Pre-filled from tcab1_pdx_sv_calls.csv (Lawlor 2026, Figure 2)
[prefill_pcgp_sv_architecture] filled 34 PCGP samples

Pre-filled SV architecture for 34 PCGP samples (Chen 2014 S4):
        sample_id sv_architecture_17p13_1
81  SJOS001103_D1                 Neither
82  SJOS001105_D1                 Neither
83  SJOS001106_D1                 Neither
84  SJOS001107_M1                 Neither
85  SJOS001107_M2                 Neither
86  SJOS001112_M2                 Neither
87  SJOS001117_D1             

## Completeness check

In [8]:
print('=== Column null / unknown rates ===')
for col in manifest.columns:
    n_unknown = (manifest[col].astype(str).str.lower() == 'unknown').sum()
    n_null    = manifest[col].isna().sum()
    if n_unknown + n_null > 0:
        print(f'  {col}: {n_unknown} unknown, {n_null} null  (out of {len(manifest)})')

=== Column null / unknown rates ===
  has_rna: 80 unknown, 0 null  (out of 115)
  has_wes: 80 unknown, 0 null  (out of 115)
  has_wgs: 46 unknown, 0 null  (out of 115)
  has_sc: 80 unknown, 0 null  (out of 115)
  has_clinical: 46 unknown, 0 null  (out of 115)
  tp53_status: 38 unknown, 0 null  (out of 115)
  tp53_evidence: 7 unknown, 0 null  (out of 115)
  tcab1_rna_quantile: 81 unknown, 0 null  (out of 115)
  tcab1_rna_value: 81 unknown, 0 null  (out of 115)
  tcab1_rna_quantile_within_cohort: 81 unknown, 0 null  (out of 115)
  tcab1_genomic_status: 115 unknown, 0 null  (out of 115)
  tcab1_genomic_evidence: 80 unknown, 0 null  (out of 115)
  tcab1_functional_status: 64 unknown, 0 null  (out of 115)
  tcab1_functional_evidence: 63 unknown, 0 null  (out of 115)
  tcab1_overall_status: 106 unknown, 0 null  (out of 115)
  tcab1_confidence: 80 unknown, 0 null  (out of 115)
  sv_architecture_17p13_1: 77 unknown, 0 null  (out of 115)
  sv_architecture_evidence: 42 unknown, 0 null  (out of 1

## Write manifest

In [9]:
MANIFEST_PATH = ROOT / 'data' / 'manifests' / 'master_manifest.csv'
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
manifest.to_csv(MANIFEST_PATH, index=False)
print(f'Manifest written: {MANIFEST_PATH}  ({len(manifest)} rows)')

Manifest written: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-alt-osteo pipeline/data/manifests/master_manifest.csv  (115 rows)


## Notebook summary

In [10]:
cohort_counts = manifest['cohort'].value_counts().to_dict()
report_lines = [
    '# Notebook 01 — Build Manifest Report',
    '',
    f'Run: {RUN_TIMESTAMP}',
    '',
    f'## Cohort counts',
    *[f'- {k}: {v}' for k, v in cohort_counts.items()],
    '',
    f'Total rows: {len(manifest)}',
    '',
    '## Input checksums',
    *[f'- {k}: {v[:12]}…' for k, v in input_checksums.items()],
    '',
    '## What changed',
    '- master_manifest.csv created with all auto-fill columns.',
    '- SV architecture pre-filled from Lawlor 2026 for 5 PDX samples.',
    '- tcab1_genomic_status, alt_status remain unknown pending Phase 3/4.',
]
report_path = ROOT / 'data' / 'processed' / 'notebook_01_report.md'
report_path.write_text('\n'.join(report_lines))
print('\n'.join(report_lines))

# Notebook 01 — Build Manifest Report

Run: 2026-05-01T17:00:11Z

## Cohort counts
- DepMap_26Q1: 35
- PCGP_2014: 34
- de_Nonneville_2022: 22
- Lawlor_2026_GCCRI: 17
- Taglialatela_2026: 7

Total rows: 115

## Input checksums
- Model.csv: ea4e0b2a3bc8…
- OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv: 0377be80c525…
- OmicsSomaticMutationsMatrixDamaging.csv: 45709b1b842d…
- OmicsCNGeneWGS.csv: d2fe65b51680…

## What changed
- master_manifest.csv created with all auto-fill columns.
- SV architecture pre-filled from Lawlor 2026 for 5 PDX samples.
- tcab1_genomic_status, alt_status remain unknown pending Phase 3/4.
